# USA — Already-on-BFP APT Validation

## Purpose

Validate a set of **scoped APT ids** (USA) against the current layer `14533` and today's
Building Footprint (BFP), then emit the Data Correctness (DC) input format for the ones that
qualify.

## What this notebook does

1. **Existence check** — for each given scoped APT id, check whether the id still **exists in
   layer `14533`** (the latest APT snapshot).
2. **On-BFP check** — for the existing ids, check whether the APT's location falls **on today's
   Building Footprint (BFP)** (spatial intersection of the APT point with the current BFP polygons).
3. **Build DC input** — for the qualifying APTs, produce the **DC input format** CSV used to
   trigger the Data Correctness process.


In [ ]:
%sql
DROP DATABASE IF EXISTS usa_alredy_on_bfp_validation CASCADE;
CREATE DATABASE IF NOT EXISTS usa_alredy_on_bfp_validation;

In [ ]:
%scala
// Shared constants used across the cells below (defined once here; values do not change between runs).
val LAYER_ID = 14533
val COUNTRY_ISO = "USA"
val latestSnapshotTable = "usa_alredy_on_bfp_validation"

In [ ]:
%scala
// Step 2 - Load the LATEST snapshot into a Delta table (filtered to country = USA).
// The newest revision is resolved automatically via LoadSnapshotInfo.getLatestRevisionId.

import com.databricks.dbutils_v1.DBUtilsHolder
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.addressing.bulk.commons.model.LayerVersions
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import com.tomtom.addressing.bulk.scala.load.LoadFreshSnapshotData
import com.tomtom.orbis.addressing.bulk.commons.odp.LoadSnapshotInfo
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository
import org.apache.spark.sql.functions._

val mapper = new ObjectMapper()
  .configure(DeserializationFeature.FAIL_ON_UNKNOWN_PROPERTIES, false)
  .configure(DeserializationFeature.ACCEPT_EMPTY_STRING_AS_NULL_OBJECT, true)
  .configure(DeserializationFeature.ACCEPT_SINGLE_VALUE_AS_ARRAY, true)

val env = "PROD"
implicit val sparky = spark
ConfigLoader.forEnvironment(env.toLowerCase)

val latestRevision = LoadSnapshotInfo.getLatestRevisionId(LAYER_ID)
println(s"Loading LATEST snapshot revision: ${latestRevision.get}")
val versionsBuilder = LayerVersions.builder()
versionsBuilder.layer(LAYER_ID, latestRevision.get)
val versionMetadata: String = mapper.writeValueAsString(versionsBuilder.build())
DBUtilsHolder.dbutils.widgets.text("layer-versions", versionMetadata)

SparkHelper.init(NEW_DATABASE)
new LoadFreshSnapshotData().run()

// Persist the loaded latest snapshot, keeping only APTs whose metadata:country tag = ESP
val latestAptDS = new OrbisElementRepository(LAYER_ID.toString).readAll
  .filter(exists(col("tags"), t =>
    t.getField("tagKey").getField("key") === "metadata:country"
    && t.getField("value") === COUNTRY_ISO))
latestAptDS.write.format("delta").mode("overwrite").saveAsTable(latestSnapshotTable)
println(s"LATEST snapshot written to: $latestSnapshotTable")